In [1]:
import pandas as pd
import numpy as np

In [2]:
files = ['BTC.parquet', 'Tesla.parquet', 'Apple.parquet', 'QQQ.parquet', 'spx.parquet']

In [9]:
h = [3, 5, 10, 30, 60]
k = [1, 2, 3, 5, 10, 15, 20]

In [18]:
for i in files:
    f = pd.read_parquet(i)
    close = f["Close"].iloc[:, 0]
    high = f["High"].iloc[:, 0]
    open_ = f["Open"].iloc[:, 0]
    low = f["Low"].iloc[:, 0]
    volume = f['Volume'].iloc[:,0]
    df = pd.DataFrame(index=f.index)
    logret_1 = np.log(close / close.shift(1))
    for j in h:
        df[f"target_{j}"] = np.log(close.shift(-j) / close)
        df[f"logret_{j}"] = np.log(close / close.shift(j))
        df[f'dist_sma{j}'] = close / close.rolling(j).mean() - 1
        df[f'vol{j}'] = logret_1.rolling(j).std()
        df[f'volume_ratio{j}'] = volume / volume.rolling(j).mean()
    df['vol_ratio'] = logret_1.rolling(5).std()/ logret_1.rolling(20).std()
    df['drawdown'] = close / close.rolling(252).max() - 1
    df['range_pct'] = (high - low) / close
    df["body"] = (close - open_) / open_
    df["close_location"] = (close - low) / (high - low)
    df['upper_shadow'] = (high - np.maximum(open_,close))/close
    df['lower_shadow'] = (np.minimum(open_,close)-low)/close
    df["dist_sma200"] = (close / close.rolling(200).mean() - 1)
    df["momentum_accel"] = np.log(close / close.shift(20)) - np.log(close / close.shift(60))
    df["parkinson_vol"] = np.sqrt((np.log(high / low) ** 2)/(4 * np.log(2)))
    df["abs_ret_1"] = np.abs(logret_1)
    df["streak"] = (np.sign(logret_1).groupby((np.sign(logret_1) != np.sign(logret_1.shift())).cumsum()).cumcount() + 1) * np.sign(logret_1)
    df = df.dropna()
    new_file = i.replace(".parquet", "_features.parquet")
    df.to_parquet(new_file)

In [19]:
df.head()

,target_3,logret_3,dist_sma3,vol3,volume_ratio3,target_5,logret_5,dist_sma5,vol5,volume_ratio5,...,range_pct,body,close_location,upper_shadow,lower_shadow,dist_sma200,momentum_accel,parkinson_vol,abs_ret_1,streak
Date,,,,,,,,,,,,,,,,,,,,,
2015-09-16,-0.014305,0.017319,0.010040,0.008789,1.103467,-0.028751,0.027060,0.013901,0.006241,1.086012,...,0.009688,0.008741,0.899123,0.000977,0.000045,-0.035823,0.012290,0.005841,0.008668,2.0
2015-09-17,-0.024136,0.018853,0.001174,0.007930,1.135426,-0.029555,0.019232,0.007423,0.007196,1.211070,...,0.017149,-0.002571,0.101669,0.012828,0.001744,-0.038145,0.021215,0.010229,0.002564,-1.0
2015-09-18,-0.009890,-0.010193,-0.011685,0.012503,1.305587,-0.013725,-0.001541,-0.008558,0.011469,1.499629,...,0.018493,-0.015897,0.126486,0.000000,0.002339,-0.053445,0.035160,0.011030,0.016296,-2.0
2015-09-21,-0.017814,-0.014305,-0.002416,0.010599,0.727904,-0.044281,0.007112,-0.005436,0.011407,0.803481,...,0.012120,0.003126,0.468538,0.006441,0.002562,-0.048876,0.064552,0.007276,0.004555,1.0
2015-09-22,-0.005885,-0.024136,-0.006735,0.011085,0.872197,-0.030654,-0.018032,-0.014163,0.010684,0.910487,...,0.016559,-0.009509,0.420267,0.000000,0.006959,-0.060299,0.104373,0.009932,0.012395,-1.0


In [20]:
df.shape

(2631, 37)